Dataset Load

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Saving railway_delay_analyzer_finaldataset to railway_delay_analyzer_finaldataset
Dataset loaded successfully.
Dataset shape: (50000, 40)


Overall Results

In [2]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

total_records = len(df)

delayed_records = (df["delay"] > 0).sum()

delay_over_15 = (df["delay"] > 15).sum()

on_time_records = (df["delay"] == 0).sum()

average_delay = df["delay"].mean()

maximum_delay = df["delay"].max()

minimum_delay = df["delay"].min()

print("===== OVERALL DELAY RESULTS =====")

print("Total Records:", total_records)
print("On-Time Records:", on_time_records)
print("Delayed Records:", delayed_records)
print("Delay > 15 Minutes:", delay_over_15)
print("Average Delay:", round(average_delay, 2), "minutes")
print("Maximum Delay:", maximum_delay, "minutes")
print("Minimum Delay:", minimum_delay, "minutes")

===== OVERALL DELAY RESULTS =====
Total Records: 50000
On-Time Records: 5119
Delayed Records: 44444
Delay > 15 Minutes: 35619
Average Delay: 87.45 minutes
Maximum Delay: 3335.0 minutes
Minimum Delay: -76.0 minutes


Delay Percentage

In [3]:
delayed_percentage = (delayed_records / total_records) * 100

high_delay_percentage = (delay_over_15 / total_records) * 100

print("===== DELAY PERCENTAGES =====")

print(
    "Delayed Record Percentage:",
    round(delayed_percentage, 2),
    "%"
)

print(
    "Delay > 15 Minutes Percentage:",
    round(high_delay_percentage, 2),
    "%"
)

===== DELAY PERCENTAGES =====
Delayed Record Percentage: 88.89 %
Delay > 15 Minutes Percentage: 71.24 %


Train-wise Result

In [4]:
train_results = (
    df.groupby("train_no")
      .agg(
          total_records=("delay", "count"),
          average_delay=("delay", "mean"),
          maximum_delay=("delay", "max"),
          total_delay=("delay", "sum")
      )
      .reset_index()
)

top_delayed_trains = (
    train_results
    .sort_values("average_delay", ascending=False)
    .head(10)
)

print("===== TOP 10 TRAINS BY AVERAGE DELAY =====")

print(
    top_delayed_trains[
        ["train_no", "total_records", "average_delay", "maximum_delay"]
    ].to_string(index=False)
)

===== TOP 10 TRAINS BY AVERAGE DELAY =====
 train_no  total_records  average_delay  maximum_delay
     1022             46     348.826087          561.0
     1054            611     312.851064         1181.0
     1021             46     297.760870          533.0
     1030            127     195.417323          778.0
     1053            884     176.634615          886.0
     1026           3637     164.222436         3335.0
     1028           5130     147.910526         1930.0
     1140           2718     139.423841          649.0
     1025           3658     133.443412          784.0
     1139           2746     127.187181          535.0


Station-wise Result

In [5]:
station_results = (
    df.groupby(["station_no", "station_name"])
      .agg(
          total_records=("delay", "count"),
          average_delay=("delay", "mean"),
          maximum_delay=("delay", "max")
      )
      .reset_index()
)

top_delayed_stations = (
    station_results
    .sort_values("average_delay", ascending=False)
    .head(10)
)

print("===== TOP 10 STATIONS BY AVERAGE DELAY =====")

print(
    top_delayed_stations[
        [
            "station_no",
            "station_name",
            "total_records",
            "average_delay",
            "maximum_delay"
        ]
    ].to_string(index=False)
)

===== TOP 10 STATIONS BY AVERAGE DELAY =====
 station_no station_name  total_records  average_delay  maximum_delay
         10         PCOI              2     648.000000          651.0
         25          LTT              3     589.000000         1130.0
         24          TNA             26     575.923077         1181.0
         23          KYN             26     533.961538         1160.0
         21           NK             26     471.538462         1141.0
         12         PCOI              9     451.888889         1573.0
         22          IGP             26     442.384615         1100.0
         14          PPI             26     418.884615         1050.0
          4         ARMU              2     408.500000          561.0
         13           NU             26     408.500000         1034.0


Special Train Result

In [6]:
special_mask = (
    df["type_code"].astype(str).str.contains(
        "SPECIAL",
        case=False,
        na=False
    )
    |
    df["train_name"].astype(str).str.contains(
        r"\bSPL\b|SPECIAL",
        case=False,
        na=False,
        regex=True
    )
)

special_trains = df[special_mask]

print("===== SPECIAL TRAIN RESULTS =====")

print("Special Train Records:", len(special_trains))

if len(special_trains) > 0:

    print(
        "Average Special Train Delay:",
        round(special_trains["delay"].mean(), 2),
        "minutes"
    )

    print(
        "Maximum Special Train Delay:",
        special_trains["delay"].max(),
        "minutes"
    )

else:
    print("No special-train records found.")

===== SPECIAL TRAIN RESULTS =====
Special Train Records: 34751
Average Special Train Delay: 52.38 minutes
Maximum Special Train Delay: 1762.0 minutes


Weather Result

In [7]:
weather_results = (
    df.groupby("weather_condition")
      .agg(
          total_records=("delay", "count"),
          average_delay=("delay", "mean")
      )
      .reset_index()
      .sort_values("average_delay", ascending=False)
)

print("===== WEATHER-WISE RESULTS =====")

print(weather_results.to_string(index=False))

===== WEATHER-WISE RESULTS =====
weather_condition  total_records  average_delay
         Fog/Mist           1841     103.134709
     Cool & Clear           1600      95.869375
            Clear          16830      91.014439
       Light Rain            759      90.673254
    Partly Cloudy           5122      87.584147
        Hot & Dry           1689      87.522795
           Cloudy          10822      86.362133
       Heavy Rain           2102      80.049477
             Rain           6125      79.546286
     Humid/Cloudy           3110      77.887781


Congestion Result

In [8]:
congestion_results = (
    df.groupby("congestion_level")
      .agg(
          total_records=("delay", "count"),
          average_delay=("delay", "mean")
      )
      .reset_index()
      .sort_values("average_delay", ascending=False)
)

print("===== CONGESTION-WISE RESULTS =====")

print(congestion_results.to_string(index=False))

===== CONGESTION-WISE RESULTS =====
congestion_level  total_records  average_delay
          Severe          10035     257.635476
            High          12294      93.966732
        Moderate          26139      23.993726
             Low           1532       3.161880


Generate Final Results Table

In [9]:
final_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "On-Time Records",
        "Delayed Records",
        "Delay > 15 Minutes",
        "Delayed Percentage",
        "Delay > 15 Percentage",
        "Average Delay",
        "Maximum Delay",
        "Minimum Delay",
        "Special Train Records"
    ],
    "Value": [
        total_records,
        on_time_records,
        delayed_records,
        delay_over_15,
        round(delayed_percentage, 2),
        round(high_delay_percentage, 2),
        round(average_delay, 2),
        maximum_delay,
        minimum_delay,
        len(special_trains)
    ]
})

print("===== FINAL PROJECT RESULTS =====")
print(final_summary.to_string(index=False))

===== FINAL PROJECT RESULTS =====
               Metric    Value
        Total Records 50000.00
      On-Time Records  5119.00
      Delayed Records 44444.00
   Delay > 15 Minutes 35619.00
   Delayed Percentage    88.89
Delay > 15 Percentage    71.24
        Average Delay    87.45
        Maximum Delay  3335.00
        Minimum Delay   -76.00
Special Train Records 34751.00


Save Final Results

In [10]:
final_summary.to_csv(
    "stage8_final_results.csv",
    index=False
)

top_delayed_trains.to_csv(
    "stage8_top_delayed_trains.csv",
    index=False
)

top_delayed_stations.to_csv(
    "stage8_top_delayed_stations.csv",
    index=False
)

weather_results.to_csv(
    "stage8_weather_results.csv",
    index=False
)

congestion_results.to_csv(
    "stage8_congestion_results.csv",
    index=False
)

print("All Stage 8 result files saved successfully.")

All Stage 8 result files saved successfully.


Interpretation

In [11]:
print("========== RESULTS & INTERPRETATION ==========\n")

print(
    f"1. The dataset contains {total_records:,} railway records."
)

print(
    f"2. {delayed_records:,} records were identified as delayed, "
    f"representing {delayed_percentage:.2f}% of all records."
)

print(
    f"3. {delay_over_15:,} records had delays greater than "
    f"15 minutes ({high_delay_percentage:.2f}% of all records)."
)

print(
    f"4. The overall average delay was "
    f"{average_delay:.2f} minutes."
)

print(
    f"5. The maximum recorded delay was "
    f"{maximum_delay:.2f} minutes."
)

print(
    f"6. {len(special_trains):,} records were identified as "
    f"special-train records."
)

print(
    "7. Train-wise and station-wise aggregation was used "
    "to identify records with higher average delays."
)

print(
    "8. Weather and congestion conditions were compared "
    "with average railway delays."
)

print(
    "9. Monthly visualization was used to observe "
    "changes in railway delays over time."
)

print(
    "\nOverall, the analysis provides a structured view of "
    "railway delay patterns across time, trains, stations, "
    "special trains, weather conditions, and congestion levels."
)

========== RESULTS & INTERPRETATION ==========

1. The dataset contains 50,000 railway records.
2. 44,444 records were identified as delayed, representing 88.89% of all records.
3. 35,619 records had delays greater than 15 minutes (71.24% of all records).
4. The overall average delay was 87.45 minutes.
5. The maximum recorded delay was 3335.00 minutes.
6. 34,751 records were identified as special-train records.
7. Train-wise and station-wise aggregation was used to identify records with higher average delays.
8. Weather and congestion conditions were compared with average railway delays.
9. Monthly visualization was used to observe changes in railway delays over time.

Overall, the analysis provides a structured view of railway delay patterns across time, trains, stations, special trains, weather conditions, and congestion levels.
